<a id="mc-overview"></a>

$$
\text{蒙特卡洛方法 (Monte Carlo Method): 用大量随机抽样来近似计算确定的量} \\
\text{(概率, 期望, 积分, 复杂系统的性能指标), 核心是 "把要算的量写成某个随机变量的期望"} \\ \quad \\
\text{理论基础 1: 大数定律 (Law of Large Numbers, LLN)} \\
X_1, \cdots, X_n \text{ 独立同分布, } E[X] = \mu, \ Var(X) = \sigma^2 \implies \bar{X}_n = \frac{1}{n}\sum X_i \to \mu \\ \quad \\
\text{推导 (弱大数定律): 先证切比雪夫不等式 (Chebyshev's Inequality)} \\
\text{对非负随机变量 Y 和 a > 0 (马尔可夫不等式 Markov's Inequality): } E[Y] \geq E[Y \cdot 1_{Y \geq a}] \geq a P(Y \geq a) \implies P(Y \geq a) \leq \frac{E[Y]}{a} \\
\text{取 } Y = (\bar{X}_n - \mu)^2, \ a = \varepsilon^2 \text{, 且 } Var(\bar{X}_n) = \frac{\sigma^2}{n} \text{:} \\
P(|\bar{X}_n - \mu| \geq \varepsilon) \leq \frac{E[(\bar{X}_n - \mu)^2]}{\varepsilon^2} = \frac{\sigma^2}{n \varepsilon^2} \to 0 \\ \quad \\
\text{理论基础 2: 中心极限定理, 误差的大小} \\
\bar{X}_n - \mu \approx N\left(0, \frac{\sigma^2}{n}\right) \implies \text{误差} \sim \frac{\sigma}{\sqrt{n}} \text{, 95\% 置信区间 } \bar{X}_n \pm 1.96 \frac{S}{\sqrt{n}} \\
\text{精度提高 10 倍, 样本数要增加 100 倍; 但误差和维数无关 (这是蒙特卡洛在高维问题中的优势)}
$$

数学基础: [期望和方差的性质 (Var(均值) = σ²/n)](../math_basics/statistics.ipynb#expectation-variance-properties) · [中心极限定理](../math_basics/statistics.ipynb#clt) · [置信区间](../math_basics/statistics.ipynb#confidence-interval)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.integrate import quad
from scipy.special import erf

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

rng = np.random.default_rng(0)
dice = rng.integers(1, 7, size=20000)
running_mean = np.cumsum(dice) / np.arange(1, len(dice) + 1)
print(f'掷骰子的期望 = 3.5, 前 10 次平均 = {running_mean[9]:.4f}, 前 20000 次平均 = {running_mean[-1]:.4f}')

# 切比雪夫上界 vs 实际概率: P(|均值 - 3.5| >= 0.1)
sigma2 = np.var(np.arange(1, 7))       # 35/12
for n in [100, 1000, 5000]:
    means = rng.integers(1, 7, size=(5000, n)).mean(axis=1)
    print(f'n = {n:>4}: 实际 P(|均值 - 3.5| >= 0.1) = {np.mean(np.abs(means - 3.5) >= 0.1):.4f}, 切比雪夫上界 = {min(1, sigma2 / (n * 0.01)):.4f}')

plt.figure(figsize=(4, 3))
plt.semilogx(np.arange(1, len(dice) + 1), running_mean, label='running mean')
plt.axhline(3.5, color='r', linestyle='--', label='E[X] = 3.5')
plt.xlabel('n')
plot_nicely()
plt.show()

<a id="estimate-pi"></a>

$$
\text{例: 估计 } \pi \\
\text{在正方形 } [-1, 1]^2 \text{ 中均匀撒点, 落在单位圆内的概率 = 面积之比:} \\
p = \frac{\pi \cdot 1^2}{2 \times 2} = \frac{\pi}{4} \implies \pi = 4p \\
\text{令 } X_i = 1 \text{ (点在圆内) 或 0, 是伯努利变量, } E[X_i] = p \text{, 所以 } \hat{\pi} = 4 \bar{X}_n \\
\text{误差: } Var(\hat{\pi}) = 16 Var(\bar{X}_n) = \frac{16 p (1 - p)}{n} \implies \text{标准差} = \frac{4\sqrt{p(1 - p)}}{\sqrt{n}} \approx \frac{1.64}{\sqrt{n}}
$$

In [ ]:
rng = np.random.default_rng(1)
p_true = np.pi / 4
ns = np.array([100, 1000, 10000, 100000])
emp_std = []
for n in ns:
    pts = rng.uniform(-1, 1, size=(200, n, 2))
    pi_hat = 4 * np.mean(np.sum(pts**2, axis=2) <= 1, axis=1)
    emp_std.append(pi_hat.std())
    print(f'n = {n:>6}: 一次估计 = {pi_hat[0]:.5f}, 200 次估计的标准差 = {pi_hat.std():.5f}, 理论 4 sqrt(p(1-p)/n) = {4 * np.sqrt(p_true * (1 - p_true) / n):.5f}')

pts = rng.uniform(-1, 1, size=(2000, 2))
inside = np.sum(pts**2, axis=1) <= 1
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].scatter(*pts[inside].T, s=2, label='inside')
axs[0].scatter(*pts[~inside].T, s=2, label='outside')
axs[0].set_aspect('equal')
axs[0].set_title(f'pi estimate = {4 * inside.mean():.4f}')
axs[1].loglog(ns, emp_std, 'o-', label='empirical std')
axs[1].loglog(ns, 4 * np.sqrt(p_true * (1 - p_true) / ns), 'k--', label='1.64 / sqrt(n)')
axs[1].set_xlabel('n')
plot_nicely()
plt.show()

<a id="mc-integration"></a>

$$
\text{蒙特卡洛积分 (Monte Carlo Integration)} \\
\text{把积分写成期望: } U \sim \text{Uniform}(a, b) \text{, 概率密度 } \frac{1}{b - a} \\
I = \int_a^b f(x) \, dx = (b - a) \int_a^b f(x) \frac{1}{b - a} dx = (b - a) \, E[f(U)] \\
\hat{I}_n = \frac{b - a}{n} \sum_{i=1}^n f(U_i) \\ \quad \\
\text{无偏 (Unbiased): } E[\hat{I}_n] = \frac{b - a}{n} \cdot n E[f(U)] = I \\
\text{方差: } Var(\hat{I}_n) = \frac{(b - a)^2}{n} Var(f(U)) \implies \text{标准误差 } SE = (b - a) \frac{\sigma_f}{\sqrt{n}} \text{, 用样本标准差估计 } \sigma_f \\ \quad \\
\text{和确定性的数值积分比较:} \\
\text{一维: 梯形法误差 } O(h^2) = O(n^{-2}) \text{, 辛普森 } O(n^{-4}) \text{, 远好于蒙特卡洛的 } O(n^{-1/2}) \\
\text{d 维: 每个方向 m 个点的网格需要 } n = m^d \text{ 个点, 梯形法误差 } O(m^{-2}) = O(n^{-2/d}) \text{, 维数一高就很差} \\
\text{蒙特卡洛的误差仍然是 } O(n^{-1/2}) \text{, 和维数无关 (d > 4 时就比梯形法好)} \\ \quad \\
\text{例: } I_d = \int_{[0,1]^d} e^{-\|x\|^2} dx = \left(\int_0^1 e^{-t^2} dt\right)^d = \left(\frac{\sqrt{\pi}}{2} \text{erf}(1)\right)^d \text{ (被积函数可以分离, 有精确值)}
$$

数学基础: [数值积分 (梯形法, 辛普森法)](../math_basics/calculus.ipynb#numerical-integral) · [高斯积分与换元](../math_basics/calculus.ipynb#multiple-integral)

In [ ]:
rng = np.random.default_rng(2)

# 1. 一维: int_0^pi sin(x) dx = 2
n = 100000
u = rng.uniform(0, np.pi, n)
fx = np.sin(u)
I_hat = np.pi * fx.mean()
SE = np.pi * fx.std(ddof=1) / np.sqrt(n)
print(f'int_0^pi sin x dx: 蒙特卡洛 = {I_hat:.5f} ± {1.96 * SE:.5f} (95% CI), quad = {quad(np.sin, 0, np.pi)[0]:.5f}')
print(f'理论标准误差 pi * sqrt(Var(sin U)) / sqrt(n) = {np.pi * np.sqrt(0.5 - (2 / np.pi)**2) / np.sqrt(n):.5f}, 样本估计 = {SE:.5f}')
# Var(sin U) = E[sin^2 U] - E[sin U]^2 = 1/2 - (2/pi)^2

# 覆盖率检查: 1000 次独立估计, 95% 置信区间包含真值 2 的比例
u_many = rng.uniform(0, np.pi, size=(1000, 2000))
f_many = np.sin(u_many)
est = np.pi * f_many.mean(axis=1)
se = np.pi * f_many.std(axis=1, ddof=1) / np.sqrt(2000)
print(f'1000 次 (每次 n = 2000) 的 95% 置信区间覆盖真值的比例 = {np.mean(np.abs(est - 2) <= 1.96 * se):.3f}')

# 2. 高维: 每个维度上的积分
one_dim = np.sqrt(np.pi) / 2 * erf(1)
print(f'\n一维 int_0^1 e^(-t^2) dt = {one_dim:.6f}, quad = {quad(lambda t: np.exp(-t**2), 0, 1)[0]:.6f}')
budget = 100000
for d in [1, 2, 5, 10]:
    exact = one_dim**d
    x = rng.random((budget, d))
    mc = np.mean(np.exp(-np.sum(x**2, axis=1)))
    m = int(round(budget ** (1 / d)))                     # 同样的点数预算, 每个方向 m 个点
    g = np.linspace(0, 1, m)
    w = np.full(m, 1 / (m - 1)); w[[0, -1]] /= 2           # 梯形法权重
    trap = (np.sum(w * np.exp(-g**2)))**d                  # 可分离, 网格积分 = 一维梯形的 d 次方
    print(f'd = {d:>2}: 精确 = {exact:.6f}, 蒙特卡洛误差 = {abs(mc - exact):.2e}, 梯形网格 ({m} 点/维) 误差 = {abs(trap - exact):.2e}')

<a id="importance-sampling"></a>

$$
\text{重要性抽样 (Importance Sampling): 在 "重要" 的地方多抽样, 再用权重修正} \\
\text{要算 } I = E_p[f(X)] = \int f(x) p(x) \, dx \text{, 换一个容易抽样的分布 q:} \\
I = \int f(x) \frac{p(x)}{q(x)} q(x) \, dx = E_q\left[f(X) \frac{p(X)}{q(X)}\right] = E_q[f(X) w(X)], \quad w(x) = \frac{p(x)}{q(x)} \ (\text{要求 } p > 0 \text{ 处 } q > 0) \\
\hat{I}_{IS} = \frac{1}{n} \sum_{i=1}^n f(Y_i) w(Y_i), \quad Y_i \sim q \text{, 无偏 (就是上面这个期望)} \\ \quad \\
\text{方差: } Var_q(f w) = E_q[f^2 w^2] - I^2 = \int \frac{f^2 p^2}{q} dx - I^2 \\
\text{最优的 q: 若 } f \geq 0 \text{, 取 } q^*(x) = \frac{f(x) p(x)}{I} \text{, 则 } f w = f \frac{p I}{f p} = I \text{ 是常数, 方差为 0} \\
\text{(实际中不知道 I, 但说明了方向: q 要和 } f p \text{ 的形状相似)} \\ \quad \\
\text{例: 正态分布的尾部概率 } P(Z > 4) = E_p[1_{Z > 4}] \approx 3.17 \times 10^{-5} \\
\text{直接抽样: 10 万个样本里平均只有 3 个落在 Z > 4, 估计非常不准} \\
\text{取 } q = N(4, 1) \text{ (把抽样中心移到 4): } w(x) = \frac{\phi(x)}{\phi(x - 4)} = \frac{e^{-x^2/2}}{e^{-(x - 4)^2/2}} = e^{-4x + 8}
$$

In [ ]:
rng = np.random.default_rng(3)
exact_tail = stats.norm.sf(4)
n = 100000

def plain_mc(rng):
    return np.mean(rng.standard_normal(n) > 4)

def importance_mc(rng):
    y = rng.normal(4, 1, n)
    return np.mean((y > 4) * np.exp(-4 * y + 8))

plain = np.array([plain_mc(rng) for _ in range(200)])
imp = np.array([importance_mc(rng) for _ in range(200)])
print(f'精确值 P(Z > 4) = {exact_tail:.4e}')
print(f'直接抽样:    200 次估计的均值 = {plain.mean():.4e}, 标准差 = {plain.std():.2e}, 估计为 0 的次数 = {np.sum(plain == 0)}')
print(f'重要性抽样:  200 次估计的均值 = {imp.mean():.4e}, 标准差 = {imp.std():.2e}')
print(f'方差减小了 {plain.var() / imp.var():.0f} 倍')

# 理论方差: 直接抽样 p(1-p)/n; 重要性抽样 (E_q[f^2 w^2] - I^2) / n, 用数值积分算 E_q[f^2 w^2] = int_4^inf w(x) phi(x) dx
second_moment = quad(lambda x: np.exp(-4 * x + 8) * stats.norm.pdf(x), 4, np.inf)[0]
print(f'理论标准差: 直接 = {np.sqrt(exact_tail * (1 - exact_tail) / n):.2e}, 重要性 = {np.sqrt((second_moment - exact_tail**2) / n):.2e}')

<a id="buffon"></a>

$$
\text{布丰投针 (Buffon's Needle): 地上画着间距为 d 的平行线, 随机扔一根长 } l \leq d \text{ 的针, 和线相交的概率} \\ \quad \\
\text{描述针的位置: y = 针的中点到最近一条线的距离, } y \sim U(0, \frac{d}{2}) \text{; } \theta \text{ = 针和线的夹角, } \theta \sim U(0, \frac{\pi}{2}) \text{, 两者独立} \\
\text{针在垂直方向上的半长是 } \frac{l}{2} \sin\theta \text{, 相交} \iff y \leq \frac{l}{2} \sin\theta \\
P = \int_0^{\pi/2} \int_0^{\frac{l}{2}\sin\theta} \underbrace{\frac{2}{d} \cdot \frac{2}{\pi}}_{\text{联合密度}} \, dy \, d\theta = \frac{4}{\pi d} \int_0^{\pi/2} \frac{l}{2} \sin\theta \, d\theta = \frac{2l}{\pi d} \left[-\cos\theta\right]_0^{\pi/2} = \frac{2l}{\pi d} \\
\implies \pi \approx \frac{2 l n}{d \cdot k} \quad (\text{扔 n 次, 相交 k 次}) \\ \quad \\
\text{模拟时要避免 "用 } \pi \text{ 来估计 } \pi \text{": 不直接抽 } \theta \text{, 而是在单位圆内均匀取一点 (u, v) (拒绝抽样), 它的方向是均匀的, } \sin\theta = \frac{|v|}{\sqrt{u^2 + v^2}}
$$

In [ ]:
rng = np.random.default_rng(4)
l, d = 0.8, 1.0
n = 1000000

# 在单位圆内均匀取方向 (拒绝抽样, 不需要 pi)
uv = rng.uniform(-1, 1, size=(int(n * 1.4), 2))
uv = uv[np.sum(uv**2, axis=1) <= 1][:n]
sin_theta = np.abs(uv[:, 1]) / np.linalg.norm(uv, axis=1)
y = rng.uniform(0, d / 2, n)
k = np.sum(y <= l / 2 * sin_theta)
p_hat = k / n
print(f'相交频率 = {p_hat:.5f}, 理论 2l / (pi d) = {2 * l / (np.pi * d):.5f}')
print(f'pi 的估计 = 2 l n / (d k) = {2 * l * n / (d * k):.5f}')
# 误差传递 (delta method): pi_hat = c / p_hat, 标准差约 c / p^2 * sqrt(p(1-p)/n)
p = 2 * l / (np.pi * d)
print(f'pi 估计的理论标准差 = {2 * l / d / p**2 * np.sqrt(p * (1 - p) / n):.5f}')

<a id="queue"></a>

$$
\text{例: 排队系统模拟 (Queueing Simulation)} \\
\text{顾客按泊松过程到达 (到达率 } \lambda \text{, 到达间隔} \sim Exp(\lambda) \text{), 一个服务台, 服务时间} \sim Exp(\mu) \text{, 先到先服务 (M/M/1)} \\ \quad \\
\text{1. 模拟: Lindley 递推. 设 } W_n \text{ 是第 n 个顾客的排队等待时间, } S_n \text{ 服务时间, } A_{n+1} \text{ 是第 n 和 n+1 个顾客的到达间隔} \\
\text{第 n 个顾客在到达后 } W_n + S_n \text{ 离开; 第 n+1 个顾客在 } A_{n+1} \text{ 后到达} \\
\text{如果他到的时候第 n 个还没走, 要等剩下的 } W_n + S_n - A_{n+1} \text{, 否则不用等} \\
W_{n+1} = \max(0, \ W_n + S_n - A_{n+1}) \\ \quad \\
\text{2. 理论 (生灭过程 Birth-Death Process): 设 } \pi_k \text{ 是系统中有 k 个人的稳态概率, } \rho = \frac{\lambda}{\mu} < 1 \\
\text{平衡方程: 稳态时, 从 k 变到 k+1 的速率 = 从 k+1 变回 k 的速率} \\
\lambda \pi_k = \mu \pi_{k+1} \implies \pi_{k+1} = \rho \pi_k \implies \pi_k = \rho^k \pi_0 \\
\sum_k \pi_k = \pi_0 \frac{1}{1 - \rho} = 1 \implies \pi_k = (1 - \rho) \rho^k \text{ (几何分布)} \\
\text{平均人数 } L = \sum_k k (1 - \rho) \rho^k = \frac{\rho}{1 - \rho} \\
\text{利特尔法则 (Little's Law): 平均人数 = 到达率} \times \text{平均逗留时间, } L = \lambda W \implies W = \frac{1}{\mu - \lambda} \\
\text{排队等待时间 = 逗留时间 - 服务时间: } W_q = W - \frac{1}{\mu} = \frac{\lambda}{\mu(\mu - \lambda)} \\
\rho \to 1 \text{ 时 } W_q \to \infty \text{: 服务能力接近满负荷时, 排队会急剧变长}
$$

数学基础: [泊松分布](../math_basics/statistics.ipynb#poisson) · [指数分布 (等待时间, 无记忆性)](../math_basics/statistics.ipynb#exponential)

In [ ]:
def simulate_mm1(lam, mu, n_customers, rng):
    A = rng.exponential(1 / lam, n_customers)      # 到达间隔
    S = rng.exponential(1 / mu, n_customers)       # 服务时间
    W = np.zeros(n_customers)
    for i in range(n_customers - 1):
        W[i + 1] = max(0.0, W[i] + S[i] - A[i + 1])   # Lindley 递推
    return W, S

rng = np.random.default_rng(5)
mu = 1.0
rhos = [0.3, 0.5, 0.7, 0.8, 0.9]
sim_wq = []
for rho in rhos:
    lam = rho * mu
    W, S = simulate_mm1(lam, mu, 300000, rng)
    W = W[10000:]                                  # 去掉开始的暂态 (warm-up)
    theory = lam / (mu * (mu - lam))
    sim_wq.append(W.mean())
    # 不用等 = 到达时系统是空的; 泊松到达的顾客看到的状态分布等于时间平均 (PASTA), 所以概率 = pi_0 = 1 - rho
    print(f'rho = {rho}: 模拟平均等待 = {W.mean():.4f}, 理论 W_q = {theory:.4f}, 不用等的比例 = {np.mean(W == 0):.4f} (理论 1 - rho = {1 - rho:.1f})')

rho_grid = np.linspace(0.05, 0.92, 100)
plt.figure(figsize=(4, 3))
plt.plot(rho_grid, rho_grid / (1 - rho_grid), label='theory W_q = rho / (mu (1 - rho))')
plt.plot(rhos, sim_wq, 'ro', label='simulation')
plt.xlabel('utilization rho = lambda / mu')
plt.ylabel('mean waiting time')
plot_nicely()
plt.show()